In [1]:
import sys
import os
import shutil
import platform
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt;
import pyemu
import flopy
#assert "dependencies" in flopy.__file__
#assert "dependencies" in pyemu.__file__
sys.path.insert(0,"..")
import herebedragons as hbd

`MODFLOW 6 & PESTPP-GLM` + executables `bin_new` 

In [2]:
org_d = os.path.join('..', '..', 'models', 'monthly_model_files_1lyr_newstress')
tmp_d = os.path.join('freyberg_mf6')
if os.path.exists(tmp_d):
    shutil.rmtree(tmp_d)
shutil.copytree(org_d,tmp_d)
hbd.prep_bins(tmp_d)        # get executables                 _____ org_d  ____   tmp_d _____
hbd.prep_deps(tmp_d)        # get dependency folders

`PstFrom` class provides methods for processing these file types into a PEST(++) dataset. 

In [ ]:
os.listdir(tmp_d)

In [3]:
sim = flopy.mf6.MFSimulation.load(sim_ws=tmp_d)           # simulation          ####   *_ws
gwf = sim.get_model()                                     # flow model
pyemu.os_utils.run("mf6",cwd=tmp_d)                       # run ______ in 'freyberg_mf6' folder
pyemu.os_utils.run(r'mp7 freyberg_mp.mpsim', cwd=tmp_d)   # run modpath7 

loading simulation...
  loading simulation name file...
  loading tdis package...
  loading model gwf6...
    loading package dis...
    loading package ic...
    loading package npf...
    loading package sto...
    loading package oc...
    loading package wel...
    loading package rch...
    loading package ghb...
    loading package sfr...
    loading package obs...
  loading solution package freyberg6...
mf6.exe
mp7.exe freyberg_mp.mpsim


`PstFrom | SpatialReference` to parameters alocation ____ `Flopy modelgrid` (alternative)

In [4]:
sr = pyemu.helpers.SpatialReference.from_namfile(
        os.path.join(tmp_d, "freyberg6.nam"),
        delr=gwf.dis.delr.array, delc=gwf.dis.delc.array)
# sr

   could not remove start_datetime


In [6]:
template_ws = os.path.join("freyberg6_template")           # _____template_ws _______
start_datetime="1-1-2008"
pf = pyemu.utils.PstFrom(original_d=tmp_d,                 # where the model is stored                         | PstFrom |      pf = PROJECT
                            new_d=template_ws,             # the PEST template folder    _______  new_d
                            remove_existing=True,          # ensures a clean start
                            longnames=True,                # set False if using PEST/PEST_HP
                            spatial_reference=sr,          # the spatial reference we generated earlier
                            zero_based=False,              # does the MODEL use zero based indices? For example, MODFLOW does NOT
                            start_datetime=start_datetime, # required when specifying temporal correlation between parameters
                            echo=False)                    # to stop PstFrom from writting lots of infromation to the notebook; # True # False

In [ ]:
# os.listdir(template_ws)

In [7]:
for i in gwf.obs:                                          # Where it is stored ?
    print(i.output.obs_names)

['sfr.csv']
['heads.csv']


In [8]:
df = pd.read_csv(os.path.join(template_ws,"heads.csv"),index_col=0)
df.head()

,TRGW-0-2-15,TRGW-0-2-9,TRGW-0-3-8,TRGW-0-9-1,TRGW-0-13-10,TRGW-0-15-16,TRGW-0-21-10,TRGW-0-22-15,TRGW-0-24-4,TRGW-0-26-6,TRGW-0-29-15,TRGW-0-33-7,TRGW-0-34-10
time,,,,,,,,,,,,,
3652.5,34.626504,35.594707,35.715488,36.718945,34.797358,34.378744,34.713368,34.239559,35.452834,35.066278,34.166173,34.636158,34.341889
3683.5,34.633560,35.608653,35.728759,36.732080,34.768494,34.359508,34.679023,34.205825,35.436127,35.008396,34.156127,34.604404,34.285080
3712.5,34.684382,35.696183,35.818154,36.833304,34.821230,34.399718,34.726236,34.240511,35.500678,35.055096,34.195098,34.654121,34.328380
3743.5,34.756157,35.829617,35.956444,36.997418,34.920983,34.469788,34.822798,34.303634,35.624531,35.166286,34.258479,34.754157,34.416088
3773.5,34.813334,35.948797,36.081709,37.157083,35.008383,34.526045,34.909527,34.354764,35.744619,35.274165,34.308607,34.844030,34.490486


In [9]:
hds_df = pf.add_observations("heads.csv",                       # ADD
                            insfile="heads.csv.ins",            # heads.csv.ins
                            index_cols="time",                  # TIME
                            use_cols=list(df.columns.values),   # columns.values
                            prefix="hds")                       # PREFIX ____________ + Weights

In [10]:
df = pd.read_csv(os.path.join(template_ws, "sfr.csv"), index_col=0)
df.head()

,HEADWATER,TAILWATER,GAGE-1
time,,,
3652.5,-1271.609373,-934.744886,2675.694849
3683.5,-1188.261301,-903.583105,2561.700004
3712.5,-1278.176409,-975.062935,2720.180988
3743.5,-1442.004051,-1100.483634,3005.022456
3773.5,-1576.905331,-1201.101468,3237.266916


In [11]:
sfr_df = pf.add_observations("sfr.csv", 
                            insfile="sfr.csv.ins", 
                            index_cols="time", 
                            use_cols=list(df.columns.values), 
                            prefix="sfr")                        # PREFIX ____________ + Weights

In [ ]:
E:\15_REPOS\00_BETAMI\01_GMDSI_____advanced\01_Tutorials___AGAIN___\part2_01_pstfrom______W_in_progress\freyberg6_template\heads.csv.ins
E:\15_REPOS\00_BETAMI\01_GMDSI_____advanced\01_Tutorials___AGAIN___\part2_01_pstfrom______W_in_progress\freyberg6_template\sfr.csv.ins

In [12]:
sfr_df.head() # hds_df.head()

,obsnme,obsval,weight,obgnme
oname:sfr_otype:lst_usecol:gage-1_time:3652.5,oname:sfr_otype:lst_usecol:gage-1_time:3652.5,2675.694849,1.0,oname:sfr_otype:lst_usecol:gage-1
oname:sfr_otype:lst_usecol:gage-1_time:3683.5,oname:sfr_otype:lst_usecol:gage-1_time:3683.5,2561.700004,1.0,oname:sfr_otype:lst_usecol:gage-1
oname:sfr_otype:lst_usecol:gage-1_time:3712.5,oname:sfr_otype:lst_usecol:gage-1_time:3712.5,2720.180988,1.0,oname:sfr_otype:lst_usecol:gage-1
oname:sfr_otype:lst_usecol:gage-1_time:3743.5,oname:sfr_otype:lst_usecol:gage-1_time:3743.5,3005.022456,1.0,oname:sfr_otype:lst_usecol:gage-1
oname:sfr_otype:lst_usecol:gage-1_time:3773.5,oname:sfr_otype:lst_usecol:gage-1_time:3773.5,3237.266916,1.0,oname:sfr_otype:lst_usecol:gage-1


In [ ]:
# [f for f in os.listdir(template_ws) if f.endswith(".ins")]

`FREYBERG` 1 layer, transient, Kh, Kv, Ss, Sy varies in space. RCH varies over both space and time. _____ GHB parameters are constant over time, but vary spatially. _____ SFR inflow varies over time._____ WEL pumping rates of individual wells are uncertain in space and and time. (not found yet).

`02 PAR geostatistical structures.` Let's assume that heterogeneity for grid-scale parameters has a shorter correlation length than pilot point parameters. 

In [13]:
v_grid = pyemu.geostats.ExpVario(contribution=1.0,      # sill  
                                    a=1000,             # range 
                                    anisotropy=1.0,    
                                    bearing=0.0         # angle in degrees East of North corresponding to anisotropy ellipse
                                    )
grid_gs = pyemu.geostats.GeoStruct(variograms=v_grid, transform='log')          # _______Grid                   # ax = grid_gs.plot()                                                     

In [14]:
v_pp = pyemu.geostats.ExpVario(contribution=1.0,        
                                    a=15000,           
                                    anisotropy=1.0,    
                                    bearing=0.0         
                                    )
pp_gs = pyemu.geostats.GeoStruct(variograms=v_pp, transform='log')             # _____ Pilot Points             # _ = pp_gs.plot() 

In [15]:
v_time = pyemu.geostats.ExpVario(contribution=1.0,       
                                    a=60, 
                                    anisotropy=1.0, 
                                    bearing=0.0 
                                    )
temporal_gs = pyemu.geostats.GeoStruct(variograms=v_time, transform='none')    # _______Time

### `Add Parameters GRID`

In [16]:
tag = "npf_k_"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
print(files)

['freyberg6.npf_k_layer1.txt']


Scales = dials that PEST(++) can turn to improve the fit. ___ Coarse constant; Medium pilot points; Finest grid.
 - `upper_bound` and `lower_bound` __________________________ Standard control file bounds 
 - `ult_ubound` and `ult_lbound`   _____________________________ Last limit 

In [17]:
ib = gwf.dis.idomain.array[0]     # idomain array      # plt.imshow(ib)

In [18]:
f = 'freyberg6.npf_k_layer1.txt'
df_gr = pf.add_parameters(f,                                        # grid (fine) scale parameters         add_parameters  df_gr
                zone_array=ib,                                      # exclude inactive model cells
                par_type="grid",                                    # Finnest
                geostruct=grid_gs,           
                par_name_base=f.split('.')[1].replace("_","")+"gr", # Name base "_gr" for "grid", ...
                pargp=f.split('.')[1].replace("_","")+"gr",         # Group 
                lower_bound=0.2, upper_bound=5.0,                   # 
                ult_ubound=100, ult_lbound=0.01                     # Multiplied. _____ Kh cannot exceed these values.multipliers !
                )                                                   # We will write the PEST control file later on.

E:\15_REPOS\00_BETAMI\01_GMDSI_____advanced\01_Tutorials___AGAIN___\part2_01_pstfrom______W_in_progress\freyberg6_template\npfklayer1gr_inst0_grid.csv.tpl

In [19]:
df_gr.tail()   # *.tpl  # *.zone 

,parnme,parval1,i,j,x,y,pargp,tpl_filename,input_filename,partype,partrans,parubnd,parlbnd,scale,offset,parchglim,dercom
pname:npfklayer1gr_inst:0_ptype:gr_pstyle:m_i:39_j:11_x:2875.00_y:125.00_zone:1,pname:npfklayer1gr_inst:0_ptype:gr_pstyle:m_i:39_j:11_x:2875.00_y:125.00_zone:1,1.0,39,11,2875.0,125.0,npfklayer1gr,freyberg6_template\npfklayer1gr_inst0_grid.csv.tpl,freyberg6_template\mult\npfklayer1gr_inst0_grid.csv,grid,log,5.0,0.2,1.0,0.0,factor,1
pname:npfklayer1gr_inst:0_ptype:gr_pstyle:m_i:39_j:12_x:3125.00_y:125.00_zone:1,pname:npfklayer1gr_inst:0_ptype:gr_pstyle:m_i:39_j:12_x:3125.00_y:125.00_zone:1,1.0,39,12,3125.0,125.0,npfklayer1gr,freyberg6_template\npfklayer1gr_inst0_grid.csv.tpl,freyberg6_template\mult\npfklayer1gr_inst0_grid.csv,grid,log,5.0,0.2,1.0,0.0,factor,1
pname:npfklayer1gr_inst:0_ptype:gr_pstyle:m_i:39_j:13_x:3375.00_y:125.00_zone:1,pname:npfklayer1gr_inst:0_ptype:gr_pstyle:m_i:39_j:13_x:3375.00_y:125.00_zone:1,1.0,39,13,3375.0,125.0,npfklayer1gr,freyberg6_template\npfklayer1gr_inst0_grid.csv.tpl,freyberg6_template\mult\npfklayer1gr_inst0_grid.csv,grid,log,5.0,0.2,1.0,0.0,factor,1
pname:npfklayer1gr_inst:0_ptype:gr_pstyle:m_i:39_j:14_x:3625.00_y:125.00_zone:1,pname:npfklayer1gr_inst:0_ptype:gr_pstyle:m_i:39_j:14_x:3625.00_y:125.00_zone:1,1.0,39,14,3625.0,125.0,npfklayer1gr,freyberg6_template\npfklayer1gr_inst0_grid.csv.tpl,freyberg6_template\mult\npfklayer1gr_inst0_grid.csv,grid,log,5.0,0.2,1.0,0.0,factor,1
pname:npfklayer1gr_inst:0_ptype:gr_pstyle:m_i:39_j:15_x:3875.00_y:125.00_zone:1,pname:npfklayer1gr_inst:0_ptype:gr_pstyle:m_i:39_j:15_x:3875.00_y:125.00_zone:1,1.0,39,15,3875.0,125.0,npfklayer1gr,freyberg6_template\npfklayer1gr_inst0_grid.csv.tpl,freyberg6_template\mult\npfklayer1gr_inst0_grid.csv,grid,log,5.0,0.2,1.0,0.0,factor,1


In [20]:
[f for f in os.listdir(template_ws) if f.endswith(".tpl")]  # NOT SURE IF IT writes the template, ... or just points out  

['npfklayer1gr_inst0_grid.csv.tpl']

### `Add Parameters PILOT POINTS`
`PstFrom` | `pp_space` extension ".csv" for dataframe, ".shp" for shapefile (point-type)

In [21]:
df_pp = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="pilotpoints",
                    geostruct=pp_gs,
                    par_name_base=f.split('.')[1].replace("_","")+"pp",
                    pargp=f.split('.')[1].replace("_","")+"pp",
                    lower_bound=0.2,upper_bound=5.0,
                    ult_ubound=100, ult_lbound=0.01,
                    pp_space=5)                                      # Unifrom grid of pilot points in every 4th row and column

starting interp point loop for 706 points
starting 0
starting 1
starting 2
starting 3
starting 4
starting 5
starting 6
starting 7
starting 8
starting 9
took 4.160497 seconds


In [ ]:
fig,ax = plt.subplots(1,1,figsize=(2,3))               # PLOT
ax.set_aspect("equal")
ax.pcolormesh(sr.xcentergrid, sr.ycentergrid,ib)
ax.scatter(df_pp.x,df_pp.y)

### `Add Parameters constant (coarse) parameter multiplier`

In [ ]:
df_cst = pf.add_parameters(f,                  # Beta _____ Supressed________
                    zone_array=ib,
                    par_type="constant",
                    geostruct=grid_gs,
                    par_name_base=f.split('.')[1].replace("_","")+"cn",
                    pargp=f.split('.')[1].replace("_","")+"cn",
                    lower_bound=0.2,upper_bound=5.0,
                    ult_ubound=100, ult_lbound=0.01)

In [22]:
[f for f in os.listdir(template_ws) if f.endswith(".tpl")]       # NOT SURE IF IT writes the template, ... or just points out

['npfklayer1gr_inst0_grid.csv.tpl', 'npfklayer1pp_inst0pp.dat.tpl']

In [23]:
df = pf.add_observations(f,prefix="hk",zone_array=ib)
df   

,obsnme,obsval,weight,obgnme
oname:hk_otype:arr_i:0_j:0_zone:1,oname:hk_otype:arr_i:0_j:0_zone:1,11.1,1.0,hk
oname:hk_otype:arr_i:0_j:10_zone:1,oname:hk_otype:arr_i:0_j:10_zone:1,11.1,1.0,hk
oname:hk_otype:arr_i:0_j:11_zone:1,oname:hk_otype:arr_i:0_j:11_zone:1,11.1,1.0,hk
oname:hk_otype:arr_i:0_j:12_zone:1,oname:hk_otype:arr_i:0_j:12_zone:1,11.1,1.0,hk
oname:hk_otype:arr_i:0_j:13_zone:1,oname:hk_otype:arr_i:0_j:13_zone:1,11.1,1.0,hk
...,...,...,...,...
oname:hk_otype:arr_i:9_j:1_zone:1,oname:hk_otype:arr_i:9_j:1_zone:1,11.1,1.0,hk
oname:hk_otype:arr_i:9_j:2_zone:1,oname:hk_otype:arr_i:9_j:2_zone:1,11.1,1.0,hk
oname:hk_otype:arr_i:9_j:3_zone:1,oname:hk_otype:arr_i:9_j:3_zone:1,11.1,1.0,hk
oname:hk_otype:arr_i:9_j:8_zone:1,oname:hk_otype:arr_i:9_j:8_zone:1,11.1,1.0,hk


In [24]:
def add_mult_pars(f, lb=0.2, ub=5.0, ulb=0.01, uub=100, add_coarse=True):
    if isinstance(f,str):
        base = f.split(".")[1].replace("_","")
    else:
        base = f[0].split(".")[1]
    pf.add_parameters(f,                                        
                    zone_array=ib,
                    par_type="grid", #_________________________ # grid (fine) scale parameters
                    geostruct=grid_gs,  
                    par_name_base=base+"gr", 
                    pargp=base+"gr", 
                    lower_bound=lb, upper_bound=ub, 
                    ult_ubound=uub, ult_lbound=ulb)
    pf.add_parameters(f,                                        
                        zone_array=ib,
                        par_type="pilotpoints", #______________ # pilot point (medium) scale parameters
                        geostruct=pp_gs,
                        par_name_base=base+"pp",
                        pargp=base+"pp",
                        lower_bound=lb, upper_bound=ub,
                        ult_ubound=uub, ult_lbound=ulb,
                        pp_space=5) 
    if add_coarse==True:
        pf.add_parameters(f,                                    
                            zone_array=ib,
                            par_type="constant", #______________ # constant (coarse) scale parameters
                            geostruct=grid_gs,
                            par_name_base=base+"cn",
                            pargp=base+"cn",
                            lower_bound=lb, upper_bound=ub,
                            ult_ubound=uub, ult_lbound=ulb)
    return

In [25]:
tag = "sto_ss"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
for f in files[1:]: 
    add_mult_pars(f, lb=0.2, ub=5.0, ulb=1e-7, uub=1e-3)

tag = "sto_sy"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
f = files[0]
add_mult_pars(f, lb=0.2, ub=5.0, ulb=0.01, uub=0.4)

tag = "ne_"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
for f in files: 
    add_mult_pars(f, lb=0.2, ub=5.0, ulb=0.01, uub=0.4)

In [28]:
tag = "sto_ss"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
for f in files[1:]: 
    add_mult_pars(f, lb=0.2, ub=5.0, ulb=1e-7, uub=1e-3)                              #  stored in the inter memory?

In [ ]:
# len([f for f in os.listdir(template_ws) if f.endswith(".tpl")])

In [ ]:
# [f for f in os.listdir(template_ws) if f.endswith(".tpl")]

## Spatial and Temporal Correlation
A set of parameters constant in space, varying in time __&__ a set of multiplier parameters constant in time  varying in space.

In [29]:
dts = pd.to_datetime(start_datetime) + pd.to_timedelta(np.cumsum(sim.tdis.perioddata.array["perlen"]),unit='d')
dts   # the container

DatetimeIndex(['2017-12-31 12:00:00', '2018-01-31 12:00:00',
               '2018-03-01 12:00:00', '2018-04-01 12:00:00',
               '2018-05-01 12:00:00', '2018-06-01 12:00:00',
               '2018-07-01 12:00:00', '2018-08-01 12:00:00',
               '2018-09-01 12:00:00', '2018-10-01 12:00:00',
               '2018-11-01 12:00:00', '2018-12-01 12:00:00',
               '2019-01-01 12:00:00', '2019-02-01 12:00:00',
               '2019-03-01 12:00:00', '2019-04-01 12:00:00',
               '2019-05-01 12:00:00', '2019-06-01 12:00:00',
               '2019-07-01 12:00:00', '2019-08-01 12:00:00',
               '2019-09-01 12:00:00', '2019-10-01 12:00:00',
               '2019-11-01 12:00:00', '2019-12-01 12:00:00',
               '2020-01-01 12:00:00'],
              dtype='datetime64[ns]', freq=None)

### Tells `PstFrom` to broadcast (e.g. share) the same parameters for all of those files: 

In [30]:
tag = "rch_recharge"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
sp = [int(f.split(".")[1].split('_')[-1]) for f in files]
d = {s:f for s,f in zip(sp,files)}
sp.sort()
files = [d[s] for s in sp]
print(files)
add_mult_pars(files, lb=0.2, ub=5.0, ulb=0, uub=1e-3, add_coarse=False)   # the spatial multiplier parameters; just use the same function

['freyberg6.rch_recharge_1.txt', 'freyberg6.rch_recharge_2.txt', 'freyberg6.rch_recharge_3.txt', 'freyberg6.rch_recharge_4.txt', 'freyberg6.rch_recharge_5.txt', 'freyberg6.rch_recharge_6.txt', 'freyberg6.rch_recharge_7.txt', 'freyberg6.rch_recharge_8.txt', 'freyberg6.rch_recharge_9.txt', 'freyberg6.rch_recharge_10.txt', 'freyberg6.rch_recharge_11.txt', 'freyberg6.rch_recharge_12.txt', 'freyberg6.rch_recharge_13.txt', 'freyberg6.rch_recharge_14.txt', 'freyberg6.rch_recharge_15.txt', 'freyberg6.rch_recharge_16.txt', 'freyberg6.rch_recharge_17.txt', 'freyberg6.rch_recharge_18.txt', 'freyberg6.rch_recharge_19.txt', 'freyberg6.rch_recharge_20.txt', 'freyberg6.rch_recharge_21.txt', 'freyberg6.rch_recharge_22.txt', 'freyberg6.rch_recharge_23.txt', 'freyberg6.rch_recharge_24.txt', 'freyberg6.rch_recharge_25.txt']


In [31]:
# len([f for f in os.listdir(template_ws) if f.endswith(".tpl")])

16

#### Assign an additional `constant` multipliers for each recharge stress-period. Then ___ Specify temporal correlation for `these` multipliers.

In [32]:
for f in files:                        # multiplier that includes temporal correlation get the stress period number from the file name
    kper = int(f.split('.')[1].split('_')[-1]) - 1       # add the constant parameters (with temporal correlation)
    pf.add_parameters(filenames=f,
                    zone_array=ib,
                    par_type="constant",
                    par_name_base=f.split('.')[1]+"tcn",
                    pargp=f.split('.')[1]+"tcn",
                    lower_bound=0.5, upper_bound=1.5,
                    ult_ubound=1e-3, ult_lbound=0,
                    datetime=dts[kper],                  # this places the parameter value on the "time axis"
                    geostruct=temporal_gs)

..\freyberg6_template\rch_recharge_9tcn_inst0_constant.csv.tpl __________
..\freyberg6_template\rch_recharge_9tcn_inst0_constant.csv.zone

In [33]:
tag = "ghb_stress_period_data"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
print(files)

['freyberg6.ghb_stress_period_data_1.txt']


Since these boundaries are likely to be very influential, we want to include a robust representation of their uncertainty - both head and conductance and at multiple scales.  

Let's parameterize both GHB conductance and head:

 - For conductance, we shall use two scales of `multiplier` parameters (`constant` and `grid`).

 - For heads, multipliers are not ideal. Instead we will use `additive` parameters. Again, with a coarse and fine scale.

 **ATTENTION!** 
 
 Additive parameters by default get assigned an initial parameter value of zero. This can be problematic later on when computing the derivatives. Be sure to either apply a parameter offset, or use "absolute" increment types in the parameter group section (we will implement the latter option further on in the current tutorial.)

In [ ]:
tag = "ghb_stress_period_data"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]

for f in files:
    # constant and grid scale multiplier conductance parameters
    name = 'ghbcond'
    pf.add_parameters(f,
                        par_type="grid",
                        geostruct=grid_gs,
                        par_name_base=name+"gr",
                        pargp=name+"gr",
                        index_cols=[0,1,2], #column containing lay,row,col
                        use_cols=[4], #column containing conductance values
                        lower_bound=0.1,upper_bound=10.0)
    pf.add_parameters(f,
                        par_type="constant",
                        geostruct=grid_gs,
                        par_name_base=name+"cn",
                        pargp=name+"cn",
                        index_cols=[0,1,2],
                        use_cols=[4],  
                        lower_bound=0.1,upper_bound=10.0,
                        ult_lbound=0.01, ult_ubound=100) #absolute limits

    # constant and grid scale additive head parameters
    name = 'ghbhead'
    pf.add_parameters(f,
                        par_type="grid",
                        geostruct=grid_gs,
                        par_name_base=name+"gr",
                        pargp=name+"gr",
                        index_cols=[0,1,2],
                        use_cols=[3],   # column containing head values
                        lower_bound=-2.0,upper_bound=2.0,
                        par_style="a", # specify additive parameter
                        transform="none", # specify not log-transform
                        ult_lbound=32.5, ult_ubound=42) #absolute limits; make sure head is never lower than the bottom of layer1
    pf.add_parameters(f,
                        par_type="constant",
                        geostruct=grid_gs,
                        par_name_base=name+"cn",
                        pargp=name+"cn",
                        index_cols=[0,1,2],
                        use_cols=[3],
                        lower_bound=-2.0,upper_bound=2.0, 
                        par_style="a", 
                        transform="none",
                        ult_lbound=32.5, ult_ubound=42) 

Easy peasy.

Now, this will make some people uncomfortable, but how well do we really ever know historic water use flux rates in space and in time? hmmm, not really! And just a little uncertainty in historic water use can result in large changes in simulated water levels...So let's add parameters to represent that uncertainty in the model inputs.

For wells it may not (or it may...) make sense to include spatial correlation. Here we will assume temporal correlation - its reasonable that pumping rates today will be similar to pumping rates yesterday. 

Pumping rates for different stress periods are in separate files. We will call `.add_parameters()` for each file. But we want to specify correlation between parameters in different files. As explained above for recharge, we do this with the parameter group name.

OK, let's get started.


As discussed above, including temporal correlation introduces an additional challenge. We use the same approach described for recharge parameters:

 - First, we will assign a `constant` multiplier parameter for each WEL stress-period file (so, a single multiplier for all well pumping rates for each stress period). We will specify temporal correlation for these `constant` multipliers.

 - Then, we will also have `grid` type multiplier parameters for each WEL stress period file (so, multipliers for individual well pumping rate during each stress period). These will not include (temporal) correlation. (We could in principle include spatial correlation here if we wanted to; but let's not).

In [ ]:
files = [f for f in os.listdir(template_ws) if "wel_stress_period_data" in f and f.endswith(".txt")]
sp = [int(f.split(".")[1].split('_')[-1]) for f in files]
d = {s:f for s,f in zip(sp,files)}
sp.sort()
files = [d[s] for s in sp]

for f in files:
    # get the stress period number from the file name
    kper = int(f.split('.')[1].split('_')[-1]) - 1  
    
    # add the constant parameters (with temporal correlation)
    pf.add_parameters(filenames=f,
                        index_cols=[0,1,2], #columns that specify cell location
                        use_cols=[3],       #columns with parameter values
                        par_type="constant",    #each well will be adjustable
                        par_name_base="welcst",
                        pargp="welcst", 
                        upper_bound = 4, lower_bound=0.25,
                        datetime=dts[kper], # this places the parameter value on the "time axis"
                        geostruct=temporal_gs)
    
    # add the grid parameters; each individual well
    pf.add_parameters(filenames=f,
                        index_cols=[0,1,2], #columns that specify cell location 
                        use_cols=[3],       #columns with parameter values
                        par_type="grid",    #each well will be adjustable
                        par_name_base="welgrd",
                        pargp="welgrd", 
                        upper_bound = 4, lower_bound=0.25,
                        datetime=dts[kper]) # this places the parameter value on the "time axis"
                     

And finally, our favorite (not!) boundary-condition: SFR.

Let's parameterize conductance (time-invariant) and inflow (time-variant).

In [ ]:
# SFR conductance
tag = "sfr_packagedata"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
assert len(files) == 1 # There can be only one! It is tradition. Jokes.
print(files)

f = files[0]
# constant and grid scale multiplier conductance parameters
name = "sfrcond"
pf.add_parameters(f,
                par_type="grid",
                geostruct=grid_gs,
                par_name_base=name+"gr",
                pargp=name+"gr",
                index_cols=[0,2,3],
                use_cols=[9],
                lower_bound=0.1,upper_bound=10.0)
pf.add_parameters(f,
                par_type="constant",
                geostruct=grid_gs,
                par_name_base=name+"cn",
                pargp=name+"cn",
                index_cols=[0,2,3],
                use_cols=[9],
                lower_bound=0.1,upper_bound=10.0,
                ult_lbound=0.001, ult_ubound=100) #absolute limits

In [ ]:
# SFR inflow
files = [f for f in os.listdir(template_ws) if "sfr_perioddata" in f and f.endswith(".txt")]
sp = [int(f.split(".")[1].split('_')[-1]) for f in files]
d = {s:f for s,f in zip(sp,files)}
sp.sort()
files = [d[s] for s in sp]
print(files)
for f in files:
    # get the stress period number from the file name
    kper = int(f.split('.')[1].split('_')[-1]) - 1  
    # add the parameters
    pf.add_parameters(filenames=f,
                        index_cols=[0], #reach number
                        use_cols=[2],   #columns with parameter values
                        par_type="grid",    
                        par_name_base="sfrgr",
                        pargp="sfrgr", 
                        upper_bound = 10, lower_bound=0.1, #don't need ult_bounds because it is a single multiplier
                        datetime=dts[kper], # this places the parameter value on the "time axis"
                        geostruct=temporal_gs)

In [ ]:
[f for f in os.listdir(template_ws) if f.endswith(".tpl")]

Damn!  we just parameterized many recognized sources of model input uncertainty at several spatial and temporal scales.  And we expressed spatial and temporal correlation in those parameters.  One last set of parameters that we will need later for sequential data assimilation - initial conditions:

In [ ]:
files = [f for f in os.listdir(template_ws) if "ic_strt" in f and f.endswith(".txt")]
files

In [ ]:
for f in files:
    base = f.split(".")[1].replace("_","")
    df = pf.add_parameters(f,par_type="grid",par_style="d",
                      pargp=base,par_name_base=base,upper_bound=50,
                     lower_bound=15,zone_array=ib,transform="none")
    print(df.shape)

### The Forward Run Script

OK! So, we almost have all the base building blocks for a PEST(++) dataset. We have some (1) observations and some (2) parameters. We are still missing (3) the "forward run" script. Recall that in the PEST world, the "model" is not just the numerical model (e.g. MODFLOW). Instead it is a composite of the numerical model (or models) and pre- and post-processing steps, encapsulated in a "forward run" script which can be called from the command line. This command line instruction is what PEST(++) sees as "the model". During execution, PEST(++) writes values to parameter files, runs "the model", and then reads values from the observation files.

`PstFrom` automates the generation of such a script when constructing the PEST control file. The script is written to file named `forward_run.py`. It is written in Python (this is not a PEST(++) requirement, merely a convenience...we are working in Python after all...). 

How about we see that in action? Magic time! Let's create the PEST control file.

In [ ]:
pst = pf.build_pst()

Boom! Done. (Well almost.) Check the folder. You should see a new .pst file and the `forward_run.py` file. By default, the .pst file is named after the original model folder name. 

In [ ]:
[f for f in os.listdir(template_ws) if f.endswith(".py") or f.endswith(".pst") ]

We will get to the `pst` object later on (see also the "intro to pyemu" tutorial notebook). For now, let's focus on the `forward_run.py` script. It is printed out below.

This script does a bunch of things:
 - it loads necessary dependencies
 - it removes model output files to avoid the possibility of files from a previous model run being read by mistake;
 - it runs pre-processing steps (see `pyemu.helpers.apply_list_and_array_pars()`;
 - it executes system commands (usually running the simulator, i.e. MODFLOW). (*This is still missing. We will demonstrate next.*)
 - it executes post-processing steps; (*for now there aren't any*)
 - ...it washes the dishes (sorry, no it doesn't...this feature is still in development).

In [ ]:
_ = [print(line.rstrip()) for line in open(os.path.join(template_ws,"forward_run.py"))]

That's pretty amazing. But as we just saw, we still need to add commands to actually run the model.

`PstFrom` allows you to pass a list of system commands which will be executed in sequence. It also has methods for including Python functions that run before or after the system commands. These make pre-/post-processing a piece of cake. In fact, we have already started to add to it. Remember all of the multiplier and additive parameters we setup? These all require pre-processing steps to convert the PEST-generated multipliers into model input values. `PstFrom` will automatically add these functions to the `forward_run.py` script. Nifty, hey?

Next we will demonstrate how to specify the system commands and add Python functions as processing steps.

#### Sys Commands

Let's start by adding a command line instruction. These are stored as a list in `PstFrom.mod_sys_cmds`, which is currently empty. 

In [ ]:
pf.mod_sys_cmds 

To run a MODFLOW6 model from the command line, you can simply execute `mf6` in the model folder. So, we can add this command by appending it to the list. (Do this only once! Every time you append 'mf6' results in an additional call to MODFLOW6, meaning the model would be run multiple times.)

`PstFrom` will add a line to `forward_run.py` w

In [ ]:
pf.mod_sys_cmds.append("mf6") #do this only once
pf.mod_sys_cmds

We also need to run MODPATH7, so we need to add that to the list of system commands. In this case we also need to specify the modpath sim file:

In [ ]:
pf.mod_sys_cmds.append("mp7 freyberg_mp.mpsim") #do this only once
pf.mod_sys_cmds

OK, now let's re-build the Pst control file and check out the changes to the `forward_run.py` script.

You should see that `pyemu.os_utils.run(r'mf6')` has been added after the pre-processing functions.

In [ ]:
pst = pf.build_pst()

_ = [print(line.rstrip()) for line in open(os.path.join(template_ws,"forward_run.py"))]

#### Extra pre- and post-processing functions

You will also certainly need to include some additional processing steps.  These are supported through the `PstFrom.pre_py_cmds` and `PstFrom.post_py_cmds`, which are lists for pre and post model run python commands and `PstFrom.pre_sys_cmds` and `PstFrom.post_sys_cmds`, which are lists for pre and post model run system commands (these are wrapped in `pyemu.os_utils.run()`.  

But what if your additional steps are actually an entire python function? Well, we got that too! `PstFrom.add_py_function()`. This method allows you to get functions from another (pre-prepared) python source file and add them to the `forward_run.py` script. We will demonstrate this to post-process model observations after each run.

Now let's see this py-sauce in action: we are going to add a little post-processing function to extract the final simulated water level for all model cells for the last stress period from the MF6 binary headsave file and save them to ASCII format so that PEST(++) can read them with instruction files.  And, while we are at it, let's also extract the global water budget info from the MF6 listing file and store it in dataframes - these are usually good numbers to watch!  We will need the simulated water level arrays later for sequential data assimilation (wouldn't it be nice if MF6 supported the writing of ASCII format head arrays?).  Anyway, this function is stored in the "helpers.py" script which you can find in the tutorial folder.

In [ ]:
pf.add_py_function("helpers.py","extract_hds_arrays_and_list_dfs()",is_pre_cmd=False)

That last argument - `is_pre_cmd` tells `PstFrom` if the python function should be treated as a pre-processor or a post-processor. So we have added that post-processor, but we still need to setup pest observations for those ASCII head arrays.  Let's do that by first calling that function to operate once within the `template_ws` to generate the arrays and then we can add them with `add_observations()`:  

In [ ]:
import helpers
helpers.test_extract_hds_arrays(template_ws)

In [ ]:
files = [f for f in os.listdir(template_ws) if f.startswith("hdslay")]
files

In [ ]:
for f in files:
    pf.add_observations(f,prefix=f.split(".")[0],obsgp=f.split(".")[0])

In [ ]:
for f in ["inc.csv","cum.csv"]:
    df = pd.read_csv(os.path.join(template_ws,f),index_col=0)
    pf.add_observations(f,index_cols=["totim"],use_cols=list(df.columns.values),
                        prefix=f.split('.')[0],obsgp=f.split(".")[0])

Crushed it!

#### Secondary Observations

Often it is usefull to include "secondary model outcomes" as observations. These can be important components in a history-matching dataset to tease out specific aspects of system behaviour (e.g. head differences between aquifer layers to inform vertical permeabilities). Or they may be simple summaries of modelled outputs which are of interest for a prediction (e.g. minimum simulated head over a given period).

If you inspect the tutorial folder you will find a file named `helpers.py`. This is a python source file which we have prepared for you. (Open it to see how it is organized.) This is not standard `pyemu` functionality, but we provide it here as an example of incorporating custom code for observation processing. The sky's the limit! It contains a function named `process_secondary_obs()`. This function reads the model output .csv files, processes them and writes a series of new observation .csv files. These new files contain the temporal-differences between head and SFR observations. The new .csv files are named `heads.tdiff.csv`and `sfr.tdiff.csv`, respectively.

First, let's load the function here and run it so you can see what happens. (And to make sure that the observation files are in the template folder!) 

Run the next cell, then inspect the template folder. You should see several new csv files. These are the new secondary observations calculated by the post-processing function.

In [ ]:
# run the helper function
helpers.process_secondary_obs(ws=template_ws)

In [ ]:
[f for f in os.listdir(template_ws) if f.endswith(".csv")]

OK, so now let's add this function to the `forward_run.py` script.

In [ ]:
pf.add_py_function("helpers.py", # the file which contains the function
                    "process_secondary_obs(ws='.')", #the function, making sure to specify any arguments it may requrie
                    is_pre_cmd=False) # whether it runs before the model system command, or after. In this case, after.

And, boom! Bob's your uncle. As easy as that.

Now, of course we want to add these observations to `PstFrom` as well:

In [ ]:

df = pd.read_csv(os.path.join(template_ws, "sfr.tdiff.csv"), index_col=0)
_ = pf.add_observations("sfr.tdiff.csv", # the model output file to read
                            insfile="sfr.tdiff.csv.ins", #optional, the instruction file name
                            index_cols="time", #column header to use as index; can also use column number (zero-based) instead of the header name
                            use_cols=list(df.columns.values), #names of columns that include observation values; can also use column number (zero-based) instead of the header name
                            prefix="sfrtd") #prefix to all observation 
                            
df = pd.read_csv(os.path.join(template_ws, "heads.tdiff.csv"), index_col=0)
_ = pf.add_observations("heads.tdiff.csv", # the model output file to read
                            insfile="heads.tdiff.csv.ins", #optional, the instruction file name
                            index_cols="time", #column header to use as index; can also use column number (zero-based) instead of the header name
                            use_cols=list(df.columns.values), #names of columns that include observation values; can also use column number (zero-based) instead of the header name
                            prefix="hdstd") #prefix to all observation names

Remember to re-build the Pst control file:

In [ ]:
pst = pf.build_pst()

In [ ]:
_ = [print(line.rstrip()) for line in open(os.path.join(template_ws,"forward_run.py"))]

Now we see that `extract_hds_array_and_list_dfs()` has been added to the forward run script and it is being called after MF6 runs. 

In [ ]:
obs = pst.observation_data
obs

### After Building the Control File

At this point, we can do some additional modifications that would typically be done that are problem specific.  Here we can tweak the setup, specifying things such as observation weights, parameter bounds, transforms, control data, etc.

Note that any modifications made after calling `PstFrom.build_pst()` will only exist in memory - you need to call `pf.pst.write()` to record these changes to the control file on disk.  Also note that if you call `PstFrom.build_pst()` after making some changes, these changes will be lost.  

For the current case, the main thing we haven't addressed are the observations from custom *.ins files,  observation weights, parameter group INCTYP's and forecasts.

We will do so now.

#### Add Observations from INS files

Recall that we wish to include observations of particle end time and status. As mentioned earlier, MP7 output files are not ina nicely organized tabular format - so we need to construct a custom instruction file. We will do this now:

In [ ]:
# write a really simple instruction file to read the MODPATH end point file
out_file = "freyberg_mp.mpend"
ins_file = out_file + ".ins"
with open(os.path.join(template_ws, ins_file),'w') as f:
    f.write("pif ~\n")
    f.write("l7 w w w w !part_status! w w !part_time!\n")

Now add these observations to the `Pst`:

In [ ]:
pst.add_observations(ins_file=os.path.join(template_ws, ins_file),
                    out_file=os.path.join(template_ws, out_file),
                            pst_path='.')

# and then check what changed                            
obs = pst.observation_data
obs.loc[obs.obsnme=='part_status', 'obgnme'] = 'part'
obs.loc[obs.obsnme=='part_time', 'obgnme'] = 'part'

obs.iloc[-2:]

#### Parameters with Zero as Intial Value

Recall that we assigned additive parameters to the GHB heads. Our initial parameter values for these parameter types were set as 0 (zero). This creates a wee bit of trouble when calculating derivatives - this is only an issue for algorithms that use finite-difference derivatives. There are a couple of ways we could get around it. One way is to add an "offset" to the parameter initial values and to the parameter bounds. Another is to use "absolute" increment types (INCTYP). See the PEST manual or PEST++ user guides for descriptions of increment types. 

We will apply both here. 

We will assign INCTYP as 'absolute'. We will leave DERINC as 0.01 (the default). It is a reasonable value in this case.

In [ ]:
head_pargps = [i for i in pst.adj_par_groups if 'head' in i]
head_pargps

In [ ]:
pst.parameter_groups.loc[head_pargps, 'inctyp'] = 'absolute'

Now add the "offset" to parameter data entries:

In [ ]:
par = pst.parameter_data
par_names = par.loc[par.parval1==0].parnme

par.loc[par_names].head()

In [ ]:
offset = -10
par.loc[par_names, 'offset'] = offset
par.loc[par_names, ['parval1', 'parlbnd', 'parubnd']] -= offset

par.loc[par_names].head()

#### Forecasts

For most models there is a forecast/prediction that someone needs. Rather than waiting until the end of the project, the forecast should be entered into your thinking and workflow __right at the beginning__.  Here we do this explicitly by monitoring the forecasts as "observations" in the control file.  This way, for every PEST(++) analysis we do, we can watch what is happening to the forecasts - #winning

The optional PEST++ `++forecasts` control variable allows us to provide the names of one or more observations featured in the "observation data" section of the PEST control file; these are treated as predictions in FOSM predictive uncertainty analysis by PESTPP-GLM. It is also a convenient way to keep track of "forecast" observations (makes post-processing a wee bit easier later on).

Recall that, for our synthetic case we are interested in forecasting:

 - groundwater level in the upper layer at row 9 and column 1 (site named "trgw-0-9-1") in stress period 22 (time=640);
 - the "tailwater" surface-water/groundwater exchange during stress period 13 (time=367); and
 - the "headwater" surface-water/groundwater exchange at stress period 22 (time=640).
 - the particle travel time.


In [ ]:
forecasts =[
            'oname:sfr_otype:lst_usecol:tailwater_time:4383.5',
            'oname:sfr_otype:lst_usecol:headwater_time:4383.5',
            'oname:hds_otype:lst_usecol:trgw-0-9-1_time:4383.5',
            'part_time'
            ]

forecasts

In [ ]:
fobs = obs.loc[forecasts,:]
fobs

We will just set this optional pest++ argument because it will trigger certain automatic behavior later in PESTPP-GLM

In [ ]:
pst.pestpp_options['forecasts'] = forecasts

### Re-write the Control File!

Make sure to re-**write** the PEST control file. But beware, if you re-**build** the `Pst`, all these changes will be lost.

In [ ]:
pst.write(os.path.join(template_ws, 'freyberg_mf6.pst'),version=2)

So that was pretty epic. We now have a (very) high-dimensional PEST interface that includes secondary observations, as well as forecasts, ready to roll. 

If you inspect the folder, you will see PEST control file and all the necessary instruction and template files. Because we have >10k parameters, version 2 of the PEST control file was written by default. 

Shall we check that it works? Let's run PEST once (i.e. with NOPTMAX=0). Now, by default, noptmax is set to zero. But just to check:

In [ ]:
pst.control_data.noptmax

OK, so when we run PEST it will call the model once and then stop. If the next cell is successful, then everything is working. Check the folder, you should see PEST output files. (We will go into these and how to process PEST outcomes in subsequent tutorials).

In [ ]:
pyemu.os_utils.run('pestpp-glm freyberg_mf6.pst', cwd=template_ws)

Recall that we assigned observation values generated from the "base model run"? If we setup everything correctly, this means that PEST should have obtained residuals very close to zero. As mentioned, this is a good way to check for problems early on.

Let's check the Phi recorded in the *.iobj file (could also check the *.rec or *.rei files).

In [ ]:
# read the file
iobj = pd.read_csv(os.path.join(template_ws, 'freyberg_mf6.iobj'))

# check value in phi column
iobj.total_phi

 Sweet! Zero. All is well.

### Prior Parameter Covariance Matrix

One the major reasons `PstFrom` was built is to help with building the Prior - both covariance matrix and ensemble - with geostatistical correlation.  Remember all that business above related to geostatistical structures and correlations?  This is where it pays off.

Let's see how this works.  For cases with less than about 30,000 parameters, we can actually generate and visualize the prior parameter covariance matrix.  If you have more parameters, this matrix may not fit in memory.  But, not to worry, `PstFrom` has some trickery to help generate the geostatistical prior ensemble even in cases where the number of parameters is greater than 30,000. 


In [ ]:
# build the prior covariance matrix and store it as a compresed bianry file (otherwise it can get huge!)
# depending on your machine, this may take a while...
if pf.pst.npar < 35000:  #if you have more than about 35K pars, the cov matrix becomes hard to handle
    cov = pf.build_prior(fmt='coo', filename=os.path.join(template_ws,"prior_cov.jcb"))
    # and take a peek at a slice of the matrix
    try: 
        x = cov.x.copy()
        x[x==0] = np.NaN
        plt.imshow(x[:1000,:1000])
    except:
        pass

snap!  That big block must be a grid-scale parameter group...

In [ ]:
cov.row_names[:10]

And now generate a prior parameter ensemble. This step is relevant for using pestpp-ies in subsequent tutorials. Note: you do not have to call `build_prior()` before calling `draw()`!

In [ ]:
pe = pf.draw(num_reals=1000, use_specsim=True) # draw parameters from the prior distribution
pe.enforce() # enforces parameter bounds
pe.to_binary(os.path.join(template_ws,"prior_pe.jcb")) #writes the paramter ensemble to binary file
assert pe.shape[1] == pst.npar

Let's now test-run one of these geostatistical realizations (always a good idea!).  We do this by replacing the `parval1` values in the control with a row from `pe`:

In [ ]:
pst.parameter_data.loc[:,"parval1"] = pe.loc[pe.index[0],pst.par_names].values
pst.parameter_data.parval1.values

In [ ]:
pst.control_data.noptmax = 0
pst.write(os.path.join(template_ws,"test.pst"),version=2)
pyemu.os_utils.run("pestpp-glm test.pst",cwd=template_ws)

If all went well, that's it! The PEST-interface is setup, tested and we have our prior prepared. We should be good to go!

### Bonus: Understanding Multiplier-Parameters

Now the multiplier files in the `template_ws/mult` folder and the MF6 input files in the `template_ws` folder contain the values corresponding to this realization, so we can visualize the multiplier parameter process:

In [ ]:
df = pd.read_csv(os.path.join(template_ws,"mult2model_info.csv"))
kh1_df = df.loc[df.model_file.str.contains("npf_k_layer1"),:]
kh1_df

In [ ]:
org_arr = np.loadtxt(os.path.join(template_ws,kh1_df.org_file.iloc[0]))
inp_arr = np.loadtxt(os.path.join(template_ws,kh1_df.model_file.iloc[0]))
mlt_arrs = [np.loadtxt(os.path.join(template_ws,afile)) for afile in kh1_df.mlt_file]
arrs = [org_arr]
arrs.extend(mlt_arrs)
arrs.append(inp_arr)
names = ["org"]
names.extend([mf.split('.')[0].split('_')[-1] for mf in kh1_df.mlt_file])
names.append("MF6 input")
fig,axes = plt.subplots(1,kh1_df.shape[0]+2,figsize=(5*kh1_df.shape[0]+2,5))
for i,ax in enumerate(axes.flatten()):
    arr = np.log10(arrs[i])
    arr[ib==0] = np.NaN
    cb = ax.imshow(arr)
    plt.colorbar(cb,ax=ax)
    ax.set_title(names[i],loc="left")
plt.tight_layout()    
    

So now we can see the difference the pilot point geostructure makes compared to the grid-scale geostructure.  The pilot point array has a much stronger spatial correlation (over a larger distance) than the grid-scale...

# Finally, what's up with all these new files and folders? We can trace through them to see how multipliers are applied

In [ ]:
from IPython.display import Image, display
display(Image(url='./pest_flow2.gif'))